<a href="https://colab.research.google.com/github/Magdal04/Natural-Language-Processing/blob/main/LAB02_Sistem_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lucrare de laborator Nr. 2
## Sistem automat de Întrebare-Răspuns pe baza unui articol Wikipedia

> Sursa textului: pagina în limba română Wikipedia despre Taylor Swift.

### Obiective
- extragerea și curățarea unui text mai lung de 5000 de cuvinte
- preprocesare NLP cu spaCy și NLTK
- formularea a 20 de întrebări simple în limba română
- identificarea celor mai relevante fragmente și generarea răspunsurilor

> Notă: notebookul salvează un snapshot local al textului extras pentru reproductibilitate.

In [ ]:
import json
import urllib3
from pathlib import Path

from qa_system import load_or_fetch_snapshot, load_romanian_pipeline

try:
    import pandas as pd
except ImportError:
    pd = None

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

ARTICLE_TITLE = "Taylor_Swift"
ARTICLE_URL = "https://ro.wikipedia.org/wiki/Taylor_Swift"
SNAPSHOT_PATH = DATA_DIR / "taylor_swift_wikipedia_ro.txt"
QUESTION_SNAPSHOT_PATH = DATA_DIR / "qa_results_taylor_swift.json"

nlp, loaded_model_name, pos_available, ner_available = load_romanian_pipeline(try_download=True)

try:
    clean_text = load_or_fetch_snapshot(
        SNAPSHOT_PATH,
        title=ARTICLE_TITLE,
        lang="ro",
        fetch_if_missing=True,
    )
except Exception as exc:
    raise RuntimeError(
        "Nu pot încărca textul sursă. Dacă rulezi offline, pune un snapshot local în "
        f"{SNAPSHOT_PATH} (text simplu, UTF-8) și rulează din nou."
    ) from exc

word_count = len(clean_text.split())
doc = nlp(clean_text)
sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

print(f"Model spaCy folosit: {loaded_model_name}")
print(f"POS disponibil: {pos_available}")
print(f"NER disponibil: {ner_available}")
print(f"URL sursă: {ARTICLE_URL}")
print(f"Fișier snapshot: {SNAPSHOT_PATH}")
print(f"Număr total de cuvinte: {word_count}")
print(f"Număr de propoziții: {len(sentences)}")
print()
print("Primele 3 propoziții din corpus:")
for index, sentence in enumerate(sentences[:3], start=1):
    print(f"{index}. {sentence}")

if not pos_available:
    print()
    print("Atenție: modelul român preantrenat spaCy nu este instalat în acest kernel.")
    print("Notebookul rulează cu un fallback bazat pe spacy.blank('ro'), ceea ce păstrează segmentarea în propoziții, dar reduce calitatea POS și NER.")

if word_count <= 5000:
    print()
    print("Atenție: textul curățat nu depășește 5000 de cuvinte.")
    print("Sistemul QA va rula în continuare, dar poate avea acoperire redusă pentru întrebări.")

## Întrebări și strategie QA

> Mai jos definim 20 de întrebări factuale în limba română și implementăm un sistem extractiv simplu:
- detectăm tipul întrebării
- extragem cuvinte-cheie și leme
- ordonăm propozițiile după relevanță
- formulăm un răspuns scurt sau folosim propoziția cea mai potrivită

In [ ]:
from qa_system import SimpleExtractiveQA

questions = [
    "Când s-a născut Taylor Swift?",
    "Unde s-a născut Taylor Swift?",
    "Cum se numește fratele mai mic al lui Taylor Swift?",
    "La ce vârstă a început Taylor Swift să se intereseze de teatrul muzical?",
    "În ce oraș a mers Taylor Swift pentru a-și lansa cariera muzicală?",
    "Cu ce casă de discuri a semnat Taylor Swift în 2005?",
    "În ce an a fost lansat albumul de debut Taylor Swift?",
    "Cum se numește al doilea album al artistei?",
    "În ce an a fost lansat albumul Fearless?",
    "Ce album a marcat tranziția lui Taylor Swift spre pop?",
    "În ce oraș s-a mutat Taylor Swift în 2014?",
    "Cum se numește turneul care a devenit cel mai de succes turneu din istorie?",
    "În ce an a început The Eras Tour?",
    "Ce album a fost lansat pe 19 aprilie 2024?",
    "Ce album a fost lansat pe 21 octombrie 2022?",
    "Ce piesă a ocupat poziția #1 în săptămâna de debut a albumului Midnights?",
    "În ce an a semnat Taylor Swift cu Republic Records?",
    "Ce documentar despre Taylor Swift a fost lansat pe Netflix în 2020?",
    "Ce titlu i-a acordat revista Time în 2023?",
    "Câte premii Grammy are Taylor Swift?"
]

qa = SimpleExtractiveQA(nlp, sentences)

print(f"Număr de întrebări definite: {len(questions)}")
for preview_question in questions[:5]:
    print("-", preview_question)


In [ ]:
results = []

for question in questions:
    question_profile, final_answer, best_fragment, best_score = qa.answer(question, top_k=3)

    results.append(
        {
            "Întrebare": question,
            "Tip": question_profile.type,
            "Scor": best_score,
            "Fragment relevant": best_fragment,
            "Răspuns final": final_answer,
        }
    )

QUESTION_SNAPSHOT_PATH.write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Rezultatele au fost salvate în: {QUESTION_SNAPSHOT_PATH}")
print()

if pd is not None:
    results_df = pd.DataFrame(results)
    display(results_df)
else:
    for item in results:
        print(json.dumps(item, ensure_ascii=False, indent=2))

Rezultatele au fost salvate în: d:\Gorincioi_Igor\Python\Master\NLP\data\qa_results_taylor_swift.json

{
  "Întrebare": "Când s-a născut Taylor Swift?",
  "Tip": "cand",
  "Scor": 7.5,
  "Fragment relevant": "La sfârșitul lui 2019, Billboard și-a sărbătorit a 125-a aniversare, publicând „Topul Celor Mai Mari Artiști din Istorie”, unde Swift s-a clasat pe locul #8, între Michael Jackson și Stevie Wonder, fiind singura artistă din top 10 care a debutat în acest secol, dar și singura din top 10 cu mai puțin de 30 de ani de activitate, cu doar 13 de ani de carieră la acea dată.",
  "Răspuns final": "Răspuns scurt: 2019."
}
{
  "Întrebare": "Unde s-a născut Taylor Swift?",
  "Tip": "unde",
  "Scor": 8.0,
  "Fragment relevant": "La sfârșitul lui 2019, Billboard și-a sărbătorit a 125-a aniversare, publicând „Topul Celor Mai Mari Artiști din Istorie”, unde Swift s-a clasat pe locul #8, între Michael Jackson și Stevie Wonder, fiind singura artistă din top 10 care a debutat în acest secol, dar ș

## Observații finale

- sursa este extrasă automat din Wikipedia și salvată local pentru reproductibilitate
- răspunsurile sunt generate printr-un QA extractiv bazat pe ranking de propoziții
- pentru întrebările de tip unde, când și cine se încearcă întâi extragerea unui răspuns scurt
- sistemul este potrivit pentru întrebări factuale; pentru întrebări interpretative va avea limitări